# Tomato CNN - Training Notebook

Student project: train a small CNN to tell fresh tomatoes from rotten ones.

This notebook does the same thing as `2_train_model.py` in my project folder, but on Colab so I can use the GPU (training is much faster).

Steps:
1. Check the environment
2. Download the tomato dataset (FGrade)
3. Sort the photos into trained / not_trained folders
4. Build and train the CNN
5. Try the model on a few test photos
6. Save and download the model

Before running: go to Runtime -> Change runtime type -> GPU, then Runtime -> Run all.

## 1. Check the environment

Colab already has TensorFlow installed. Just check the version and that the GPU is on.

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print('GPU is on:', gpus[0].name)
else:
    print('No GPU. Training will still work, just slower.')

## 2. Download the tomato dataset

Using the FGrade dataset from GitHub. It has 6,470 photos of tomatoes labeled 1 (freshest) to 10 (most rotten). For my project I only need two classes (fresh / rotten), so I follow the paper's idea: classes 1-5 = fresh, classes 6-10 = rotten.

In [ ]:
!rm -rf /content/FGrade /content/dataset
!git clone --depth 1 https://github.com/skarifahmed/FGrade /content/FGrade --quiet
print('Done.')

## 3. Sort the photos into trained / not_trained folders

I split the 800 photos into two clearly named folders so I can talk about them in the report:

- `dataset/trained/` - 500 photos the CNN learns from (250 fresh + 250 rotten)
- `dataset/not_trained/` - 300 photos the CNN never sees during training (150 fresh + 150 rotten). Used only to test how good the model really is.

Random seed is fixed (42) so the split is always the same.

In [ ]:
import os, random, shutil
from pathlib import Path

RANDOM_SEED            = 42
TRAINED_PER_CLASS      = 250
NOT_TRAINED_PER_CLASS  = 150
FRESH_FOLDERS          = ['0','1','2','3','4']   # FGrade classes 1-5
ROTTEN_FOLDERS         = ['5','6','7','8','9']   # FGrade classes 6-10

fgrade  = Path('/content/FGrade')
dataset = Path('/content/dataset')
for sub in ['trained/fresh','trained/rotten','not_trained/fresh','not_trained/rotten']:
    (dataset/sub).mkdir(parents=True, exist_ok=True)

def collect(folders):
    out = []
    for sub in ('Training_set', 'Testing_set'):
        for c in folders:
            out.extend(sorted((fgrade/'data'/sub/c).glob('*.jpg')))
    return out

fresh_all  = collect(FRESH_FOLDERS)
rotten_all = collect(ROTTEN_FOLDERS)
print(f'Got {len(fresh_all)} fresh and {len(rotten_all)} rotten candidates.')

def split_and_save(src_paths, label, seed_offset):
    needed = TRAINED_PER_CLASS + NOT_TRAINED_PER_CLASS
    rng = random.Random(RANDOM_SEED + seed_offset)
    picks = rng.sample(src_paths, needed)
    # First TRAINED_PER_CLASS go to trained/<label>/
    for i, p in enumerate(picks[:TRAINED_PER_CLASS], 1):
        shutil.copy2(p, dataset/'trained'/label/f'{label}_{i:03d}.jpg')
    # Rest go to not_trained/<label>/
    for i, p in enumerate(picks[TRAINED_PER_CLASS:], 1):
        shutil.copy2(p, dataset/'not_trained'/label/f'{label}_{i:03d}.jpg')

# Clean any old contents first
for sub in ['trained/fresh','trained/rotten','not_trained/fresh','not_trained/rotten']:
    shutil.rmtree(dataset/sub); (dataset/sub).mkdir(parents=True)

split_and_save(fresh_all,  'fresh',  0)
split_and_save(rotten_all, 'rotten', 1)

for sub in ['trained/fresh','trained/rotten','not_trained/fresh','not_trained/rotten']:
    print(f'  dataset/{sub}: {len(list((dataset/sub).glob("*.jpg")))} photos')

## 4. Build and train the CNN

Small CNN built from scratch. Each layer has a simple job:

- Conv2D(32) + MaxPool: find edges and corners
- Conv2D(64) + MaxPool: find curves and dark spots
- Conv2D(128) + MaxPool: find tomato-level patterns
- Flatten + Dense(128) + Dropout(0.5): mix everything together
- Dense(1, sigmoid): output a number 0 to 1 (0 = fresh, 1 = rotten)

Settings: Adam optimizer with learning rate 0.001, binary cross-entropy loss, batch size 32, 15 epochs. Light data augmentation (flip, rotate, zoom) to help the small dataset go further.

In [ ]:
import numpy as np, time
from tensorflow.keras import layers, models

IMG_SIZE   = 128
BATCH_SIZE = 32
EPOCHS     = 15

random.seed(42); np.random.seed(42); tf.random.set_seed(42)

def load_folder(root):
    paths, labels = [], []
    for name, lbl in [('fresh', 0), ('rotten', 1)]:
        for p in sorted((Path(root)/name).glob('*.jpg')):
            paths.append(p); labels.append(lbl)
    X = np.zeros((len(paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    for i, p in enumerate(paths):
        img = tf.keras.utils.load_img(str(p), target_size=(IMG_SIZE, IMG_SIZE))
        X[i] = tf.keras.utils.img_to_array(img) / 255.0
    return X, np.array(labels, dtype=np.float32)

X_tr, y_tr = load_folder('/content/dataset/trained')
X_te, y_te = load_folder('/content/dataset/not_trained')

# Shuffle training set
rng = np.random.default_rng(42)
idx = np.arange(len(X_tr)); rng.shuffle(idx)
X_tr, y_tr = X_tr[idx], y_tr[idx]

print(f'Trained:     {len(X_tr)} photos')
print(f'Not trained: {len(X_te)} photos')

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()

t0 = time.time()
history = model.fit(X_tr, y_tr, validation_data=(X_te, y_te),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
print(f'\nTraining took {time.time()-t0:.1f} seconds')

loss, acc = model.evaluate(X_te, y_te, verbose=0)
print(f'\nTest accuracy on the 300 not_trained photos: {acc*100:.2f}%')

probs = model.predict(X_te, verbose=0).ravel()
preds = (probs >= 0.5).astype(np.float32)
tp = int(np.sum((y_te==1)&(preds==1))); tn = int(np.sum((y_te==0)&(preds==0)))
fp = int(np.sum((y_te==0)&(preds==1))); fn = int(np.sum((y_te==1)&(preds==0)))
print(f'\nConfusion matrix:')
print(f'  fresh  predicted fresh  : {tn}')
print(f'  fresh  predicted rotten : {fp}')
print(f'  rotten predicted fresh  : {fn}')
print(f'  rotten predicted rotten : {tp}')

## 5. Try the model on some example photos

Quick visual check that the predictions make sense. Green title = correct, red = wrong.

In [ ]:
import matplotlib.pyplot as plt

n = 8
idx = np.random.default_rng(0).choice(len(X_te), size=n, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.flat, idx):
    p = float(model.predict(X_te[i:i+1], verbose=0).ravel()[0])
    pred  = 'rotten' if p >= 0.5 else 'fresh'
    truth = 'rotten' if y_te[i] == 1 else 'fresh'
    color = 'green' if pred == truth else 'red'
    ax.imshow(X_te[i])
    ax.set_title(f'predicted: {pred} ({p*100:.0f}% rotten)\nactual: {truth}',
                 color=color, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Save and download the model

Save `tomato_model.h5` and `training_results.json`, then download both to my computer.

Where to put them after the download: drop both files into the `Final_tomatoes_cnn` folder (next to `3_server.py`). Then on my PC I just run `python 3_server.py` to use the model.

In [ ]:
import json

model.save('tomato_model.h5')

results = {
    'image_size': IMG_SIZE,
    'epochs': EPOCHS,
    'train_count': int(len(X_tr)),
    'test_count':  int(len(X_te)),
    'history': {
        'accuracy':     [float(v) for v in history.history['accuracy']],
        'val_accuracy': [float(v) for v in history.history['val_accuracy']],
        'loss':         [float(v) for v in history.history['loss']],
        'val_loss':     [float(v) for v in history.history['val_loss']],
    },
    'test_accuracy': float(acc),
    'test_loss':     float(loss),
    'confusion_matrix': {
        'true_fresh_predicted_fresh':   tn,
        'true_fresh_predicted_rotten':  fp,
        'true_rotten_predicted_fresh':  fn,
        'true_rotten_predicted_rotten': tp,
    },
}
with open('training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Saved tomato_model.h5 and training_results.json')

try:
    from google.colab import files
    files.download('tomato_model.h5')
    files.download('training_results.json')
except ImportError:
    print('Not in Colab - files saved to current folder.')

## Done

Two files downloaded:

1. `tomato_model.h5` - the trained model
2. `training_results.json` - the accuracy numbers

I drop both into my local `Final_tomatoes_cnn` folder and run:

```
pip install -r requirements.txt
python 3_server.py
```

Then the website opens at http://localhost:8000 and I can upload tomato photos to test it.